# NCFig6：路径适配与完整绘图 Notebook

本 Notebook 与 `figure6_arial_refined_commented.py` 配套使用。

- `.py`：包含全部数据读取、空间处理、绘图函数和导出流程，并带有分段注释。
- `.ipynb`：负责设置项目路径、检查输入、调用 `.py` 生成 a–h 子图及组合图，并检查输出。
- 计算和绘图参数保持原脚本不变；本次主要修复文件路径，并补充代码说明。

**运行前提：** 在 VS Code 中打开项目根目录 `NCFig6`，再打开并运行本 Notebook。

## 1. 设置项目路径与字体路径
项目根目录就是包含 `data/`、`Arial.ttf` 和 `.py` 文件的那一层。

In [ ]:
from pathlib import Path
import sys
import importlib.util
import json

# 默认当前工作目录就是项目根目录。
# 如果 VS Code 的工作目录不是项目根目录，请把下面一行改成你的实际路径。
PROJECT_DIR = Path.cwd().resolve()

# 检查项目结构，避免相对路径指向错误位置。
if not (PROJECT_DIR / "data").is_dir():
    raise FileNotFoundError(
        f"当前工作目录不是项目根目录：{PROJECT_DIR}\n"
        "请在 VS Code 中打开 NCFig6 文件夹，或手动修改 PROJECT_DIR。"
    )

SCRIPT_PATH = PROJECT_DIR / "figure6_arial_refined_commented.py"

# 字体按你的文件夹截图设置：Arial.ttf 与 .py 位于项目根目录。
# 若 Arial.ttf 在其他位置，改为 Path(r'D:/Fonts/Arial.ttf')。
FONT_FILE = PROJECT_DIR / "Arial.ttf"
FONT_BOLD_FILE = PROJECT_DIR / "arialbd.ttf"  # 粗体文件可选
ALLOW_PREVIEW_FONT = False  # 仅在没有 Arial 的临时预览环境中改为 True

if not FONT_FILE.is_file() and not ALLOW_PREVIEW_FONT:
    raise FileNotFoundError(
        f"没有找到 Arial 字体：{FONT_FILE}\n"
        "请将 Arial.ttf 放在项目根目录，或修改 FONT_FILE 为字体文件的绝对路径。"
    )
if not FONT_FILE.is_file():
    FONT_FILE = None  # 预览模式下由 Matplotlib 使用替代字体，不能作为正式 Arial 输出
if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(f"没有找到绘图脚本：{SCRIPT_PATH}")

print("项目根目录：", PROJECT_DIR)
print("Python 脚本：", SCRIPT_PATH)
print("Arial 字体：", FONT_FILE if FONT_FILE else "预览替代字体（非 Arial）")
print("data 文件夹：", PROJECT_DIR / "data")


## 2. 检查所有输入文件
特别注意：`reference_style.json` 是布局参数文件，原始截图目录里可能没有，需要从本项目包中一并复制到 `data/` 或项目根目录。

In [ ]:
# 统计数据：a–h 的 CSV。
required_files = [
    PROJECT_DIR / "data" / f"fig6{k}.csv" for k in "abcdefgh"
]

# 地图数据：Shapefile 的 .shp 必须与 .shx/.dbf/.prj 等配套文件放在一起。
required_files += [
    PROJECT_DIR / "data" / "china_admin_2024" / f"{name}.shp"
    for name in ["city", "province", "county", "south_china_sea_lines"]
]

# 版式参数 JSON：允许放在 data/ 或项目根目录。
style_candidates = [
    PROJECT_DIR / "data" / "reference_style.json",
    PROJECT_DIR / "reference_style.json",
]
required_style = next((p for p in style_candidates if p.is_file()), None)

missing = [str(p) for p in required_files if not p.is_file()]
if required_style is None:
    missing.append("reference_style.json（放在 data/ 或项目根目录）")

if missing:
    raise FileNotFoundError("以下输入文件缺失：\n" + "\n".join(missing))

print(f"统计 CSV：{8}/8")
print("四类地图边界：4/4")
print("版式参数：", required_style)


## 3. 设置绘图选项并运行完整流程
默认输出 300 dpi，分别生成 a–h 独立图和最终组合图。只想重画组合图时，将 `ONLY` 改为 `combined`。

In [ ]:
# 可修改参数
DPI = 300
ONLY = "all"       # 可选：all、combined、a、b、c、d、e、f、g、h
OUTPUT_DIR = PROJECT_DIR / "output_commented"

# 动态加载同目录的绘图脚本，避免复制粘贴整套函数。
spec = importlib.util.spec_from_file_location("fig6_plot_module", SCRIPT_PATH)
fig6 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = fig6
spec.loader.exec_module(fig6)

# 传入与命令行运行相同的参数。
args = [
    "--project-dir", str(PROJECT_DIR),
    "--dpi", str(DPI),
    "--out", str(OUTPUT_DIR),
    "--only", ONLY,
]
if FONT_FILE is not None:
    args += ["--font", str(FONT_FILE)]
if FONT_BOLD_FILE.is_file():
    args += ["--font-bold", str(FONT_BOLD_FILE)]
if ALLOW_PREVIEW_FONT:
    args += ["--allow-font-fallback"]

# 调用脚本主函数；数据读取、地图匹配、投影、绘图、导出均在 .py 中执行。
fig6.main(args)


## 4. 检查生成结果
检查独立子图、组合图、字体记录和行政代码匹配表。

In [ ]:
# 预期输出：8 张独立子图（若 ONLY='all'）+ 组合图 + 检查文件。
for ext in ["png", "pdf", "svg"]:
    p = OUTPUT_DIR / f"Fig6_combined.{ext}"
    print(ext, p.exists(), p)

validation_file = OUTPUT_DIR / "render_validation.json"
if validation_file.exists():
    validation = json.loads(validation_file.read_text(encoding="utf-8"))
    for key in [
        "font_family", "font_is_fallback", "city_matching",
        "projection", "background_layer_count", "png_mode",
        "fully_transparent_pixel_fraction"
    ]:
        print(f"{key}: {validation.get(key)}")
